In [0]:
## CELL 1 — Setup: imports and config
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import uuid
from datetime import datetime

# Team 2 config
CATALOG     = "hive_metastore"
YOUR_DB     = "team2_edulearn"
S3_BASE     = 's3://s3-de-q1-26/DE-Training/team2_edulearn_datalake/'
RUN_ID      = str(uuid.uuid4())[:8]

spark.sql(f"CREATE DATABASE IF NOT EXISTS {YOUR_DB}")
print(f"[EduLearn] Database: {YOUR_DB} | Run ID: {RUN_ID}")

In [0]:
## CELL 2 — Helper: add metadata columns
def add_metadata(df, source_name, file_name):
    """Adds 4 standard Bronze metadata columns to every table."""
    return (
        df
        .withColumn("_source",    F.lit(source_name))
        .withColumn("_ingest_ts", F.lit(datetime.utcnow().isoformat()))
        .withColumn("_file_name", F.lit(file_name))
        .withColumn("_run_id",    F.lit(RUN_ID))
        .withColumn("ingest_date", F.to_date(F.lit(datetime.utcnow().strftime("%Y-%m-%d"))))
    )


In [0]:
## CELL 3 — Ingest: enrollments_bronze
raw_path = f"{S3_BASE}/raw/enrollments/"
target_path = f"{S3_BASE}/bronze/enrollments/"

df_enroll = spark.read.option("header", "true").csv(raw_path)
df_enroll = add_metadata(df_enroll, "s3_raw_enrollments", "enrollments.csv")
df_enroll = df_enroll.filter(
    (F.col("enrollment_id").isNotNull()) &
    (F.col("total_fees").cast("double") >= 0)
)

# Enable auto-optimize
spark.conf.set("spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite", "true")

df_enroll.write \
    .format("delta") \
    .mode("append") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .partitionBy("ingest_date") \
    .saveAsTable(f"{YOUR_DB}.enrollments_bronze")

print(f"[EduLearn] enrollments_bronze: {df_enroll.count()} rows ingested")

In [0]:

## CELL 4 — Ingest: customers_bronze
raw_path = f"{S3_BASE}/raw/customers/"

df_cust = spark.read.option("header", "true").csv(raw_path)
df_cust = add_metadata(df_cust, "s3_raw_customers", "customers.csv")

df_cust.write \
    .format("delta") \
    .mode("append") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .partitionBy("ingest_date") \
    .saveAsTable(f"{YOUR_DB}.customers_bronze")

print(f"[EduLearn] customers_bronze: {df_cust.count()} rows ingested")

In [0]:
## CELL 5 — Ingest: courses_bronze
raw_path = f"{S3_BASE}/raw/courses/"

df_courses = spark.read.option("header", "true").csv(raw_path)
df_courses = add_metadata(df_courses, "s3_raw_courses", "courses.csv")

df_courses.write \
    .format("delta") \
    .mode("append") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .partitionBy("ingest_date") \
    .saveAsTable(f"{YOUR_DB}.courses_bronze")

print(f"[EduLearn] courses_bronze: {df_courses.count()} rows ingested")

In [0]:

## CELL 6 — Ingest: enrollment_items_bronze
raw_path = f"{S3_BASE}/raw/enrollment_items/"

df_items = spark.read.option("header", "true").csv(raw_path)
df_items = add_metadata(df_items, "s3_raw_enrollment_items", "enrollment_items.csv")

df_items.write \
    .format("delta") \
    .mode("append") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .partitionBy("ingest_date") \
    .saveAsTable(f"{YOUR_DB}.enrollment_items_bronze")

print(f"[EduLearn] enrollment_items_bronze: {df_items.count()} rows ingested")

In [0]:
## CELL 7 — Ingest: completions_bronze
raw_path = f"{S3_BASE}/raw/completions/"

df_comp = spark.read.option("header", "true").csv(raw_path)
df_comp = add_metadata(df_comp, "s3_raw_completions", "completions.csv")

df_comp.write \
    .format("delta") \
    .mode("append") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .partitionBy("ingest_date") \
    .saveAsTable(f"{YOUR_DB}.completions_bronze")

print(f"[EduLearn] completions_bronze: {df_comp.count()} rows ingested")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS team2_edulearn.enrollments_bronze
USING DELTA
LOCATION '{S3_BASE}/bronze/enrollments/'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS team2_edulearn.completions_bronze
USING DELTA
LOCATION '{S3_BASE}/bronze/completions/'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS team2_edulearn.courses_bronze
USING DELTA
LOCATION '{S3_BASE}/bronze/courses/'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS team2_edulearn.customers_bronze
USING DELTA
LOCATION '{S3_BASE}/bronze/customers/'
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS team2_edulearn.enrollment_items_bronze
USING DELTA
LOCATION '{S3_BASE}/bronze/enrollment_items/'
""")

In [0]:
## CELL 8 — Add CHECK constraints on enrollments_bronze
# spark.sql(f"""
#     ALTER TABLE {YOUR_DB}.enrollments_bronze
#     ADD CONSTRAINT enrollment_id_not_null
#     CHECK (enrollment_id IS NOT NULL)
# """)

# spark.sql(f"""
#     ALTER TABLE {YOUR_DB}.enrollments_bronze
#     ADD CONSTRAINT total_fees_non_negative
#     CHECK (CAST(total_fees AS DOUBLE) >= 0)
# """)

print("[EduLearn] Constraints added to enrollments_bronze")

In [0]:
# ## CELL 9 — Schema enforcement demo (intentional error)
# from pyspark.sql.types import StructType, StructField, StringType

# # This will FAIL on purpose — shows Delta's schema enforcement
# bad_df = spark.createDataFrame(
#     [("TEST_ENR", "C001", "2024-01-01", "EXTRA_VALUE")],
#     ["enrollment_id", "customer_id", "order_date", "extra_column_that_should_fail"]
# )

# try:
#     bad_df.write \
#         .format("delta") \
#         .mode("append") \
#         .saveAsTable(f"{YOUR_DB}.enrollments_bronze")
#     print("ERROR: This should have failed!")
# except Exception as e:
#     print(f"[EduLearn] Expected schema enforcement error: {type(e).__name__}")
#     print("Delta correctly rejected the extra column. Schema enforcement works!")

In [0]:
## CELL 10 — Verify: SHOW TABLES and row counts
spark.sql(f"SHOW TABLES IN {YOUR_DB}").show()

tables = [
    "enrollments_bronze", "customers_bronze", "courses_bronze",
    "enrollment_items_bronze", "completions_bronze"
]
for t in tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {YOUR_DB}.{t}").collect()[0]['cnt']
    print(f"[EduLearn] {t}: {count} rows")

In [0]:
## CELL 11 — Python Task P4: bronze_validator() function call
# (The function itself is in team2_edulearn_utils.py — paste it here for notebook use)

def bronze_validator(df, table_name, key_column, expected_columns):
    """
    Validates a Bronze Delta table DataFrame.
    Checks: row count, null %, schema completeness, metadata columns.
    """
    results = {}
    row_count = df.count()
    results['row_count'] = 'pass' if row_count > 0 else 'fail'
    print(f"\n[EduLearn Bronze Validator] ── {table_name} ──")
    print(f"  row_count       : {row_count} → {results['row_count'].upper()}")

    null_count = df.filter(
        F.col(key_column).isNull() | (F.col(key_column) == '')
    ).count()
    null_pct = (null_count / row_count * 100) if row_count > 0 else 100
    results['null_check'] = 'pass' if null_pct < 5 else 'fail'
    results['null_count'] = null_count
    print(f"  null_check      : {null_count} nulls in '{key_column}' "
          f"({null_pct:.2f}%) → {results['null_check'].upper()}")

    missing = [c for c in expected_columns if c not in df.columns]
    results['schema_check'] = 'pass' if not missing else 'fail'
    results['missing_columns'] = missing
    print(f"  schema_check    : missing={missing} → {results['schema_check'].upper()}")

    metadata_cols = ['_source', '_ingest_ts', '_file_name', '_run_id']
    meta_missing = [c for c in metadata_cols if c not in df.columns]
    results['metadata_check'] = 'pass' if not meta_missing else 'fail'
    print(f"  metadata_check  : missing_meta={meta_missing} → {results['metadata_check'].upper()}")

    return results

# Run on enrollments_bronze
df_e = spark.read.table(f"{YOUR_DB}.enrollments_bronze")
results_e = bronze_validator(
    df_e,
    table_name="enrollments_bronze",
    key_column="enrollment_id",
    expected_columns=["enrollment_id", "customer_id", "order_date", "total_fees", "enrollment_status", "city"]
)
assert results_e['row_count'] == 'pass', "FAIL: enrollments_bronze is empty!"

# Run on customers_bronze
df_c = spark.read.table(f"{YOUR_DB}.customers_bronze")
results_c = bronze_validator(
    df_c,
    table_name="customers_bronze",
    key_column="customer_id",
    expected_columns=["customer_id", "name", "city", "signup_date", "email"]
)
assert results_c['row_count'] == 'pass', "FAIL: customers_bronze is empty!"

In [0]:
## CELL 12 — Spark Task SP1: Execution Plan
df_explain = spark.read.format('delta').table(f"{YOUR_DB}.enrollments_bronze")
df_explain.filter(
    df_explain.enrollment_status == 'COMPLETED'
).select(
    'enrollment_id', 'customer_id', 'total_fees'
).explain(True)

# Paste the output into SKILLS_EVIDENCE.md under Q8

In [0]:
tables = [
    ("enrollments_bronze", df_enroll, "enrollments"),
    ("customers_bronze", df_cust, "customers"),
    ("courses_bronze", df_courses, "courses"),
    ("enrollment_items_bronze", df_items, "enrollment_items"),
    ("completions_bronze", df_comp, "completions")
]

for table_name, df, folder in tables:
    
    target_path = f"{S3_BASE}/bronze/{folder}/"
    
    # Write to S3
    df.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ingest_date") \
        .save(target_path)
    
    # Register table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {YOUR_DB}.{table_name}
        USING DELTA
        LOCATION '{target_path}'
    """)
    
    print(f"[EduLearn] {table_name} written to {target_path}")

In [0]:
%sql
SHOW TABLES IN team2_edulearn;